# QuantsMind SDK Examples
A walkthrough of capabilities that actually exist in this checkout (v1.0.1+). Every snippet below was verified by execution. Sections marked [quantum extra] need `pip install "quantsmind[quantum]"`.

## 1. Introduction
QuantsMind is a universal scientific-computing SDK: one ontology (Entity/System/State/Interaction) specialized per domain. This notebook touches mathematics, science domains, optimization, and the quantum layer end to end.

## 2. Mathematics
Value objects from `quantsmind.math`: a 3-D point and a topological space with an open set.

In [ ]:
from quantsmind.math.geometry import Point
from quantsmind.math.topology import TopologicalSpace
point = Point([1.0, 2.0, 3.0])
print(point.coordinates, point.dimension)
space = TopologicalSpace({1, 2})
space.add_open_set({1})
print(space.is_open({1}), space.is_open({2}))
# [1.0, 2.0, 3.0] 3 / True False

## 3. Calculus
Numerical Jacobian of f(x) = [x0^2, x0*x1] at (1, 2): exactly [[2, 0], [2, 1]] up to finite differences.

In [ ]:
from quantsmind.calculus.differentiation import Differentiator
jac = Differentiator().jacobian(lambda x: [x[0]**2, x[0]*x[1]], [1.0, 2.0])
print([[round(v, 3) for v in row] for row in jac])
# [[2.0, 0.0], [2.0, 1.0]]

## 4. Statistics
Standard normal moments plus a Poisson check.

In [ ]:
from quantsmind.statistics.distribution_engine import NormalDistribution, PoissonDistribution
normal = NormalDistribution(0.0, 1.0)
print(normal.mean(), normal.variance(), round(normal.pdf(0.0), 4), normal.cdf(0.0))
print(PoissonDistribution(2.0).pdf(2))
# 0.0 1.0 0.3989 0.5 / 0.2707

## 5. Physics
Projectile ballistics: 10 m/s at 45 degrees flies ~10.20 m.

In [ ]:
from quantsmind.physics import projectile_range, flight_time
print(round(projectile_range(10.0, 45.0), 2))
print(round(flight_time(10.0, 45.0), 2))
# 10.2 / 1.44

## 6. Chemistry
Water from formula to molarity.

In [ ]:
from quantsmind.chemistry import parse_formula, molar_mass, moles_from_mass, molarity
mass = molar_mass(parse_formula("H2O"))
moles = moles_from_mass(2.0 * mass, mass)
print(round(mass, 3), round(moles, 3), round(molarity(moles, 2.0), 3))
# 18.015 2.0 1.0

## 7. Biology
Transcribe and translate a 24-base fragment.

In [ ]:
from quantsmind.biology import transcribe, translate_dna, gc_content
dna = "ATGGCCATTGTAATGGGCCGCTGA"
print(transcribe(dna))
print(translate_dna(dna))
print(f"{gc_content(dna):.1%}")
# AUGGCCAUUGUAAUGGGCCGCUGA / MAIVMGR / 54.2%

## 8. Astronomy
Derive the year from 1 AU and 1 solar mass via Kepler III.

In [ ]:
from quantsmind.astronomy import ASTRONOMICAL_UNIT_M, SOLAR_MASS_KG, kepler_period
year_days = kepler_period(ASTRONOMICAL_UNIT_M, SOLAR_MASS_KG) / 86400.0
print(round(year_days, 2))
# 365.25

## 9. Cosmology
Hubble velocity at 100 Mpc and the Hubble time for H0 = 70.

In [ ]:
from quantsmind.cosmology import hubble_velocity, hubble_time_gyr, scale_factor
print(hubble_velocity(100.0))
print(round(hubble_time_gyr(), 2))
print(scale_factor(1.0))
# 7000.0 / 13.97 / 0.5

## 10. Optimization
Pick the best insulation thickness through a one-hot QUBO solved by the deterministic classical baseline.

In [ ]:
from quantsmind.physics import heat_conduction
from quantsmind.quantum.optimization.classical import ExhaustiveSolver
from quantsmind.quantum.optimization.qubo import QUBOModel
options = {"d5cm": 0.05, "d10cm": 0.10, "d15cm": 0.15, "d20cm": 0.20}
names = sorted(options)
losses = {n: heat_conduction(0.04, 10.0, 20.0, options[n]) for n in names}
model = QUBOModel(variables=names, linear=dict(losses))
result = ExhaustiveSolver().solve(model, is_feasible=lambda a: sum(a.values()) == 1)
print(result.assignment, round(result.energy, 1))
# {'d10cm': 0, 'd15cm': 0, 'd20cm': 1, 'd5cm': 0} 40.0

## 11. Quantum execution [quantum extra]
Bell state on the local statevector backend (seeded). Needs `pip install "quantsmind[quantum]"`.

In [ ]:
from quantsmind.quantum import QuantumExperiment, QuantumProgram
result = QuantumExperiment(
    QuantumProgram.bell_state(), backend="statevector", shots=1024, seed=42
).run()
print(result.success, result.counts)
# True {'11': 509, '00': 515}

## 12. Providers and execution [quantum extra]
Locate the statevector backend through the provider abstraction.

In [ ]:
from quantsmind.providers import LocalSimulatorProvider
provider = LocalSimulatorProvider()
print(provider.health())
print(provider.get_backend("statevector").capabilities().name)
# ProviderHealth(available=True, ...) / statevector

## 13. End to end
Sections 10–12 compose the documented chain: scientific problem → mathematical formulation → optimization representation → execution via providers → interpreted result. Each link above ran verbatim.